In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
results = pd.read_csv('../data/raw/results.csv')
elo = pd.read_csv('../data/raw/elo_ratings.csv')

In [5]:
results['date'] = pd.to_datetime(results['date'])
elo['snapshot_date'] = pd.to_datetime(
    elo['snapshot_date']
)

In [6]:
latest_elo = (
    elo.sort_values('snapshot_date')
       .groupby('country')
       .tail(1)
)

In [7]:
latest_elo.head()

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
4634,2026,2026-12-31,Qatar,95,QA,1425,26,1771,93,1416,...,328,146,223,289,232,176,993,825,AFC,0
4607,2026,2026-12-31,Austria,23,AT,1827,1,2070,20,1810,...,431,383,51,370,310,185,1560,1321,UEFA,0
4606,2026,2026-12-31,Paraguay,22,PY,1833,5,1956,23,1755,...,214,365,209,276,303,209,1002,1115,CONMEBOL,0
4605,2026,2026-12-31,Mexico,20,MX,1860,4,1985,20,1782,...,297,273,458,533,258,237,1823,1055,CONCACAF,1
4604,2026,2026-12-31,Belgium,19,BE,1867,1,2157,24,1755,...,415,375,86,390,302,184,1599,1359,UEFA,0


In [8]:
latest_elo = latest_elo[
    [
        'country',
        'rating',
        'rank',
        'confederation',
        'is_host'
    ]
]

In [9]:
home_elo = latest_elo.rename(columns={
    'country': 'home_team',
    'rating': 'home_elo',
    'rank': 'home_rank',
    'confederation': 'home_confederation',
    'is_host': 'home_host'
})

In [10]:
away_elo = latest_elo.rename(columns={
    'country': 'away_team',
    'rating': 'away_elo',
    'rank': 'away_rank',
    'confederation': 'away_confederation',
    'is_host': 'away_host'
})

In [11]:
results = results.merge(
    home_elo,
    on='home_team',
    how='left'
)

In [12]:
results = results.merge(
    away_elo,
    on='away_team',
    how='left'
)

In [13]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,home_rank,home_confederation,home_host,away_elo,away_rank,away_confederation,away_host
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1767.0,29.0,UEFA,0.0,2020.0,4.0,UEFA,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2020.0,4.0,UEFA,0.0,1767.0,29.0,UEFA,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1767.0,29.0,UEFA,0.0,2020.0,4.0,UEFA,0.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2020.0,4.0,UEFA,0.0,1767.0,29.0,UEFA,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1767.0,29.0,UEFA,0.0,2020.0,4.0,UEFA,0.0


In [14]:
results[['home_elo', 'away_elo']].isnull().sum()

home_elo    32195
away_elo    33522
dtype: int64

In [15]:
def get_result(row):
    
    if row['home_score'] > row['away_score']:
        return 2
    
    elif row['home_score'] < row['away_score']:
        return 0
    
    else:
        return 1

In [16]:
results['target'] = results.apply(
    get_result,
    axis=1
)

In [17]:
results['elo_diff'] = (
    results['home_elo'] - results['away_elo']
)

In [18]:
results['rank_diff'] = (
    results['away_rank'] - results['home_rank']
)

In [19]:
results['total_goals'] = (
    results['home_score'] +
    results['away_score']
)

In [20]:
results['neutral'] = results['neutral'].astype(int)

In [21]:
results['is_competitive'] = (
    results['tournament'] != 'Friendly'
).astype(int)

In [22]:
def calculate_points(row):
    
    if row['home_score'] > row['away_score']:
        return 3
    
    elif row['home_score'] == row['away_score']:
        return 1
    
    else:
        return 0

In [23]:
results['home_points'] = results.apply(
    calculate_points,
    axis=1
)

In [24]:
def calculate_away_points(row):
    
    if row['away_score'] > row['home_score']:
        return 3
    
    elif row['away_score'] == row['home_score']:
        return 1
    
    else:
        return 0

In [25]:
results['away_points'] = results.apply(
    calculate_away_points,
    axis=1
)

In [26]:
results = results.sort_values('date')

In [27]:
results['home_form'] = (
    results.groupby('home_team')['home_points']
    .transform(
        lambda x: x.rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)

In [28]:
results['away_form'] = (
    results.groupby('away_team')['away_points']
    .transform(
        lambda x: x.rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)

In [29]:
results['form_diff'] = (
    results['home_form'] -
    results['away_form']
)

In [30]:
results['home_attack_strength'] = (
    results.groupby('home_team')['home_score']
    .transform(
        lambda x: x.rolling(
            5,
            min_periods=1
        ).mean()
    )
)

In [31]:
results['away_attack_strength'] = (
    results.groupby('away_team')['away_score']
    .transform(
        lambda x: x.rolling(
            5,
            min_periods=1
        ).mean()
    )
)

In [32]:
results['home_defense_strength'] = (
    results.groupby('home_team')['away_score']
    .transform(
        lambda x: x.rolling(
            5,
            min_periods=1
        ).mean()
    )
)

In [33]:
results['away_defense_strength'] = (
    results.groupby('away_team')['home_score']
    .transform(
        lambda x: x.rolling(
            5,
            min_periods=1
        ).mean()
    )
)

In [34]:
features = [

    'home_elo',
    'away_elo',
    'elo_diff',

    'home_rank',
    'away_rank',
    'rank_diff',

    'neutral',
    'is_competitive',

    'home_form',
    'away_form',
    'form_diff',

    'home_attack_strength',
    'away_attack_strength',

    'home_defense_strength',
    'away_defense_strength',

    'elite_score_diff',

    'adjusted_strength_diff'
]

In [36]:
#ml_df = results[features + ['target']]

results['elite_score_diff'] = 0

results['adjusted_strength_diff'] = 0

In [37]:
ml_df = results[features + ['target']]

In [38]:
ml_df = ml_df.dropna()

In [39]:
ml_df.to_csv(
    '../data/cleaned/ml_dataset.csv',
    index=False
)

In [40]:
ml_df.head()

,home_elo,away_elo,elo_diff,home_rank,away_rank,rank_diff,neutral,is_competitive,home_form,away_form,form_diff,home_attack_strength,away_attack_strength,home_defense_strength,away_defense_strength,elite_score_diff,adjusted_strength_diff,target
0,1767.0,2020.0,-253.0,29.0,4.0,-25.0,0,0,1.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0,0,1
1,2020.0,1767.0,253.0,4.0,29.0,25.0,0,0,3.000000,0.000000,3.0,4.000000,2.000000,2.000000,4.000000,0,0,2
2,1767.0,2020.0,-253.0,29.0,4.0,-25.0,0,0,2.000000,0.500000,1.5,1.000000,0.500000,0.500000,1.000000,0,0,2
3,2020.0,1767.0,253.0,4.0,29.0,25.0,0,0,2.000000,0.500000,1.5,3.000000,2.000000,2.000000,3.000000,0,0,1
4,1767.0,2020.0,-253.0,29.0,4.0,-25.0,0,0,2.333333,0.333333,2.0,1.666667,0.333333,0.333333,1.666667,0,0,2


In [41]:
ml_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7464 entries, 0 to 49471
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   home_elo                7464 non-null   float64
 1   away_elo                7464 non-null   float64
 2   elo_diff                7464 non-null   float64
 3   home_rank               7464 non-null   float64
 4   away_rank               7464 non-null   float64
 5   rank_diff               7464 non-null   float64
 6   neutral                 7464 non-null   int32  
 7   is_competitive          7464 non-null   int32  
 8   home_form               7464 non-null   float64
 9   away_form               7464 non-null   float64
 10  form_diff               7464 non-null   float64
 11  home_attack_strength    7464 non-null   float64
 12  away_attack_strength    7464 non-null   float64
 13  home_defense_strength   7464 non-null   float64
 14  away_defense_strength   7464 non-null   floa